In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import os
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class PanopticDataset(Dataset):
    def __init__(self, root_dir, transform_segformer=None, transform_sam=None, max_prompts=20):
        self.root_dir = root_dir
        self.image_dir = os.path.join(root_dir, 'images')
        self.semantic_dir = os.path.join(root_dir, 'semantic_masks')
        self.instance_dir = os.path.join(root_dir, 'instance_masks')
        self.image_list = sorted(os.listdir(self.image_dir))
        self.transform_segformer = transform_segformer
        self.transform_sam = transform_sam
        self.max_prompts = max_prompts

    def __len__(self):
        return len(self.image_list)

    def __getitem__(self, idx):
        image_name = self.image_list[idx]
        image_path = os.path.join(self.image_dir, image_name)
        sem_path = os.path.join(self.semantic_dir, image_name)
        instance_folder = os.path.join(self.instance_dir, image_name.replace('.png', ''))

        image = Image.open(image_path).convert('RGB')
        semantic_mask = Image.open(sem_path)

        instance_masks = []
        categories = []
        boxes = []
        if os.path.exists(instance_folder):
            inst_files = sorted(os.listdir(instance_folder))[:self.max_prompts]
            for fname in inst_files:
                inst_path = os.path.join(instance_folder, fname)
                inst_mask = Image.open(inst_path).convert('L')
                inst_mask_np = np.array(inst_mask)
                rows, cols = np.where(inst_mask_np > 0)
                if len(rows) > 0:
                    min_row, max_row = rows.min(), rows.max()
                    min_col, max_col = cols.min(), cols.max()
                    box = [min_col, min_row, max_col + 1, max_row + 1]
                    boxes.append(box)
                instance_masks.append(inst_mask_np)
                category = int(fname.split('_')[-1].replace('class', '').replace('.png', ''))
                categories.append(category)

        if self.transform_segformer:
            image_seg = self.transform_segformer(image)
            semantic_mask = semantic_mask.resize((256, 256), resample=Image.NEAREST)
            semantic_mask = torch.tensor(np.array(semantic_mask), dtype=torch.long)
        else:
            image_seg = transforms.ToTensor()(image)

        if self.transform_sam:
            image_sam = self.transform_sam(image)
        else:
            image_sam = transforms.ToTensor()(image)

        instance_masks = [torch.from_numpy(mask).to(dtype=torch.uint8) for mask in instance_masks]
        categories = torch.tensor(categories, dtype=torch.long) if categories else torch.tensor([], dtype=torch.long)
        boxes = torch.tensor(boxes, dtype=torch.float32) if boxes else torch.tensor([], dtype=torch.float32)

        return {
            "image_seg": image_seg,
            "image_sam": image_sam,
            "semantic_mask": semantic_mask,
            "instance_masks": instance_masks,
            "categories": categories,
            "boxes": boxes,
            "image_name": image_name
        }

def custom_collate_fn(batch):
    images_seg = torch.stack([item["image_seg"] for item in batch])
    images_sam = torch.stack([item["image_sam"] for item in batch])
    semantic_masks = torch.stack([item["semantic_mask"] for item in batch])
    instance_masks = [item["instance_masks"] for item in batch]
    categories = [item["categories"] for item in batch]
    boxes = [item["boxes"] for item in batch]
    image_names = [item["image_name"] for item in batch]

    return {
        "image_seg": images_seg,
        "image_sam": images_sam,
        "semantic_mask": semantic_masks,
        "instance_masks": instance_masks,
        "categories": categories,
        "boxes": boxes,
        "image_name": image_names
    }

transform_segformer = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

transform_sam = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

dataset = PanopticDataset(
    root_dir='/kaggle/input/pannuke-preprocess-dataset/dataset',
    transform_segformer=transform_segformer,
    transform_sam=transform_sam,
    max_prompts=20
)

dataloader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=custom_collate_fn,
    num_workers=0,
    pin_memory=True if torch.cuda.is_available() else False
)

In [3]:
from transformers import SegformerForSemanticSegmentation, SegformerFeatureExtractor
import torch
from torch.optim import AdamW
from transformers import get_scheduler

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_segformer = SegformerForSemanticSegmentation.from_pretrained("nvidia/segformer-b0-finetuned-ade-512-512")
feature_extractor = SegformerFeatureExtractor.from_pretrained("nvidia/segformer-b0-finetuned-ade-512-512")
model_segformer.to(device)

optimizer_segformer = AdamW(model_segformer.parameters(), lr=5e-5)
num_epochs = 2  # Reduced for testing
num_training_steps = len(dataloader) * num_epochs
lr_scheduler = get_scheduler(
    name="linear", optimizer=optimizer_segformer, num_warmup_steps=0, num_training_steps=num_training_steps
)

model_segformer.train()
for epoch in range(num_epochs):
    total_loss = 0
    for batch in dataloader:
        images = batch["image_seg"].to(device)  # [2, 3, 256, 256]
        labels = batch["semantic_mask"].to(device)  # [2, 256, 256]

        inputs = feature_extractor(images=images, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        outputs = model_segformer(**inputs, labels=labels)
        loss = outputs.loss

        loss.backward()
        optimizer_segformer.step()
        lr_scheduler.step()
        optimizer_segformer.zero_grad()
        total_loss += loss.item()

        # Clear memory
        del inputs, outputs, loss
        torch.cuda.empty_cache()

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{num_epochs}, Average Loss: {avg_loss:.4f}")

config.json:   0%|          | 0.00/6.88k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/15.0M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/models/segformer/feature_extraction_segformer.py:28: FutureWarning: The class SegformerFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use SegformerImageProcessor instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/utils/deprecation.py:165: UserWarning: The following named arguments are not valid for `SegformerFeatureExtractor.__init__` and were ignored: 'feature_extractor_type'
  return func(*args, **kwargs)
It looks like you are trying to rescale already rescaled images. If the input images have pixel values between 0 and 1, set `do_rescale=False` to avoid rescaling them again.


Epoch 1/2, Average Loss: 0.4882
Epoch 2/2, Average Loss: 0.2948


In [4]:
from transformers import SamModel, SamProcessor
import torch.nn.functional as F
from torch.optim import Adam

model_sam = SamModel.from_pretrained("facebook/sam-vit-base")
processor = SamProcessor.from_pretrained("facebook/sam-vit-base")

for param in model_sam.vision_encoder.parameters():
    param.requires_grad = False

model_sam.to(device)
optimizer_sam = Adam([p for p in model_sam.parameters() if p.requires_grad], lr=2e-4)

num_epochs_sam = 2  # Reduced for testing
accumulation_steps = 4  # Gradient accumulation
max_prompts = 20  # Already set in dataset

model_sam.train()
for epoch in range(num_epochs_sam):
    total_loss = 0
    num_valid_samples = 0
    for sample in dataset:  # Process one sample at a time
        image = sample["image_sam"].unsqueeze(0).to(device)  # [1, 3, 256, 256]
        instance_masks = sample["instance_masks"]
        boxes = sample["boxes"].to(device)  # [N, 4]

        if len(boxes) == 0:
            continue

        # Scale boxes to 256x256
        scale_factor = 256 / 512
        boxes = boxes * scale_factor

        inputs = processor(image, return_tensors="pt", do_rescale=False)
        pixel_values = inputs["pixel_values"].to(device)  # [1, 3, 256, 256]

        try:
            outputs = model_sam(
                pixel_values=pixel_values,
                input_boxes=boxes.unsqueeze(0),  # [1, N, 4]
                multimask_output=False
            )
            pred_masks = outputs.pred_masks  # [1, N, 1, 256, 256]
        except torch.cuda.OutOfMemoryError:
            print(f"Skipping sample due to OOM: {sample['image_name']}")
            torch.cuda.empty_cache()
            continue

        losses = []
        for prompt_idx, gt_mask in enumerate(instance_masks):
            gt_mask_tensor = gt_mask.float().to(device)  # [512, 512]
            gt_mask_tensor = F.interpolate(
                gt_mask_tensor.unsqueeze(0).unsqueeze(0),  # [1, 1, 512, 512]
                size=(256, 256),
                mode='nearest'
            )[0, 0] / 255.0  # [256, 256]
            pred_mask = pred_masks[0, prompt_idx, 0]  # [256, 256]
            loss = F.binary_cross_entropy_with_logits(pred_mask, gt_mask_tensor)
            losses.append(loss)

        if losses:
            total_loss_batch = torch.stack(losses).mean() / accumulation_steps
            total_loss += total_loss_batch.item()
            total_loss_batch.backward()

            if (num_valid_samples + 1) % accumulation_steps == 0:
                optimizer_sam.step()
                optimizer_sam.zero_grad()

            num_valid_samples += 1
            del pred_masks, outputs, losses, total_loss_batch
            torch.cuda.empty_cache()

    if num_valid_samples % accumulation_steps != 0:
        optimizer_sam.step()
        optimizer_sam.zero_grad()

    if num_valid_samples > 0:
        avg_loss = total_loss / num_valid_samples
        print(f"Epoch {epoch+1}/{num_epochs_sam}, Average Loss: {avg_loss:.4f}")
    else:
        print(f"Epoch {epoch+1}/{num_epochs_sam}, No valid samples with instances")

config.json:   0%|          | 0.00/6.57k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/375M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

Epoch 1/2, Average Loss: 0.0034
Epoch 2/2, Average Loss: 0.0013


In [5]:
def panoptic_segmentation(image_path):
    from skimage.morphology import label
    import numpy as np

    image = Image.open(image_path).convert('RGB')
    input_seg = transform_segformer(image).unsqueeze(0).to(device)  # [1, 3, 256, 256]
    input_sam = transform_sam(image).unsqueeze(0).to(device)  # [1, 3, 256, 256]

    # Semantic segmentation with Segformer
    with torch.no_grad():
        inputs = feature_extractor(images=input_seg, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        outputs_seg = model_segformer(**inputs)
        semantic_labels = F.interpolate(
            outputs_seg.logits, size=(256, 256), mode='bilinear', align_corners=False
        ).argmax(dim=1)[0].cpu().numpy()  # [256, 256]

    # Instance segmentation with SAM
    panoptic_map = np.zeros_like(semantic_labels, dtype=np.int32)
    inst_id_counter = {}
    max_prompts = 20

    for category_id in range(1, semantic_labels.max() + 1):
        binary_mask = (semantic_labels == category_id).astype(np.uint8)
        labeled_mask, num_components = label(binary_mask, return_num=True)

        if category_id not in inst_id_counter:
            inst_id_counter[category_id] = 1

        box_prompts = []
        for comp_id in range(1, min(num_components + 1, max_prompts + 1)):
            component_mask = (labeled_mask == comp_id).astype(np.uint8)
            rows, cols = np.where(component_mask)
            if len(rows) == 0:
                continue
            min_row, max_row = rows.min(), rows.max()
            min_col, max_col = cols.min(), cols.max()
            box = [min_col, min_row, max_col + 1, max_row + 1]
            box_prompts.append(box)  # Already in 256x256 scale

        if not box_prompts:
            continue

        with torch.no_grad():
            inputs_sam = processor(input_sam, return_tensors="pt", do_rescale=False)
            pixel_values = inputs_sam["pixel_values"].to(device)
            outputs_sam = model_sam(
                pixel_values=pixel_values,
                input_boxes=torch.tensor(box_prompts, dtype=torch.float32).unsqueeze(0).to(device),
                multimask_output=False
            )
            pred_masks = outputs_sam.pred_masks[0]  # [N, 1, 256, 256]

        for prompt_idx in range(pred_masks.size(0)):
            pred_mask = (torch.sigmoid(pred_masks[prompt_idx, 0]) > 0.5).cpu().numpy().astype(np.uint8)
            inst_id = inst_id_counter[category_id]
            panoptic_map[pred_mask == 1] = category_id * 1000 + inst_id
            inst_id_counter[category_id] += 1

    return panoptic_map

# Test inference
image_path = '/kaggle/input/pannuke-preprocess-dataset/dataset/images/image_0000.png'
panoptic_map = panoptic_segmentation(image_path)
print(f"Panoptic map shape: {panoptic_map.shape}")

Panoptic map shape: (256, 256)
